# Sherlok
A beginner-friendly multi-agent workshop built for Google Colab. Five specialized agents investigate the fictional disappearance of the Aurora Diamond.

**Learning goals:** agent roles, prompt design, shared context, handoffs, structured outputs, and human review.

## Before you run it
1. Create a Gemini API key in Google AI Studio.
2. In Colab, open the **key icon → Secrets**.
3. Add a secret named `GEMINI_API_KEY` and enable notebook access.
4. Run every cell in order. Never paste your API key directly into the notebook.

In [ ]:
!pip -q install -U google-genai gradio

In [ ]:
from google import genai
from google.colab import userdata
import gradio as gr

API_KEY = userdata.get('GEMINI_API_KEY')
if not API_KEY:
    raise ValueError('Add GEMINI_API_KEY to Colab Secrets, then run this cell again.')

client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-2.5-flash-lite'
print('Detective headquarters is ready.')

## The fictional case
You can edit the evidence or add misleading clues before your workshop. The solution is deliberately not hard-coded—the agents must reason from the supplied facts.

In [ ]:
CASE_FILE = '''
CASE: The Vanishing Aurora Diamond
At 8:00 PM, curator Dr. Mira Sen displayed the Aurora Diamond inside a locked glass case at Northbridge Museum. At 8:20 PM, a power failure darkened the gallery for four minutes. At 8:30 PM, the diamond was gone. No glass was broken.

SUSPECTS
1. Lena Ortiz, security chief. Motive: recently denied a promotion. Says she was restarting the basement generator from 8:19–8:26. Her access card opened the basement at 8:20.
2. Theo Park, visiting magician. Motive: publicity. Says he remained on stage rehearsing. A stage camera shows him continuously from 8:15–8:29.
3. Arjun Vale, assistant curator. Motive: large private debt. Says he was cataloguing artifacts in the archive. His access card opened the archive at 8:12 and the gallery display case at 8:23.
4. Sofia Reed, journalist. Motive: wanted an exclusive story. Says she was interviewing guests in the lobby. Three guests remember speaking with her during the blackout.

EVIDENCE
A. The display case uses an electronic lock and records every valid access card, even during a power failure because it has a battery.
B. The log records Arjun's card opening the case at 8:23 PM.
C. Arjun says his access card was in his jacket inside the archive.
D. A hallway camera resumes at 8:25 and shows Arjun leaving the archive carrying a flat catalog folder.
E. Blue velvet fibers were found inside that folder. The diamond's display cushion is blue velvet.
F. A muddy shoeprint near the case matches Lena's boot size, but maintenance records show Lena inspected the same case after walking through a wet courtyard that afternoon.
G. The diamond was insured, but the policy pays the museum—not any suspect.

RULES
Use only this case file. Separate facts from inferences. Mention uncertainty. Do not invent evidence. This is a fictional educational exercise.
'''
print(CASE_FILE)

In [ ]:
AGENTS = {
    'Detective Agent': '''Build a concise case timeline. Identify the central mystery and the three most important unanswered questions. Do not decide guilt yet.''',
    'Evidence Agent': '''Evaluate every clue for reliability and relevance. Label each clue FACT, INFERENCE, or DISTRACTION. Identify the strongest evidence and explain why.''',
    'Suspect Agent': '''Compare every suspect's motive, means, opportunity, and alibi. Use a compact table. Rank suspects, but explicitly state what is not proven.''',
    'Skeptic Agent': '''Challenge the current investigation. Find alternative explanations, weak assumptions, possible planted evidence, and missing information. State what would change the conclusion.''',
    'Chief Agent': '''Act as the responsible investigation chief. Synthesize the specialist reports, name the most likely suspect, give a confidence percentage, cite the decisive clues, discuss the best alternative theory, and recommend the next investigative step. Never claim certainty beyond the evidence.''',
}

def ask_agent(name, instruction, shared_context):
    prompt = f'''You are the {name} in a multi-agent detective team.

YOUR ROLE
{instruction}

CASE FILE
{CASE_FILE}

REPORTS FROM EARLIER AGENTS
{shared_context or 'None—you are the first agent.'}

Return a clear workshop-friendly report under 250 words. Use only supplied information.'''
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

def investigate(progress=gr.Progress()):
    reports = {}
    ordered_names = list(AGENTS)
    for index, name in enumerate(ordered_names):
        progress(index / len(ordered_names), desc=f'{name} is investigating...')
        previous = '\n\n'.join(f'## {n}\n{r}' for n, r in reports.items())
        reports[name] = ask_agent(name, AGENTS[name], previous)
    progress(1, desc='Case review complete')
    return tuple(reports[name] for name in ordered_names)

print('Five agents have been created:', ', '.join(AGENTS))

In [ ]:
with gr.Blocks(title='Sherlok') as demo:
    gr.Markdown('# 🔎 Sherlok')
    gr.Markdown('Five AI agents share evidence, challenge one another, and produce a cautious verdict for a fictional case.')
    with gr.Accordion('Read the case file', open=False):
        gr.Textbox(value=CASE_FILE, lines=16, interactive=False, show_label=False)
    investigate_button = gr.Button('Start Investigation', variant='primary')
    outputs = []
    for name in AGENTS:
        outputs.append(gr.Markdown(label=name))
    investigate_button.click(fn=investigate, inputs=[], outputs=outputs)

demo.launch(share=True, debug=False)

## Workshop challenges
1. Remove clue E and compare the Chief Agent's confidence.
2. Add a new witness statement containing an inconsistency.
3. Change the Skeptic Agent's instructions and observe the verdict.
4. Ask participants whether sequential handoffs cause early-agent bias.
5. Add a Human Judge who approves or rejects the final conclusion.

**Teaching point:** Multiple agents do not guarantee truth. Their value comes from role separation, explicit evidence, criticism, and human review.